___
# <center>Curva ROC e introdução a machine learning</center>
___

## Aula 22

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * desenhar a curva ROC de um modelo e calcular a AUC;
 * separar a base em treino e teste, e comparar o erro nos dois;
 * reconhecer sobreajuste e subajuste;
 * usar validação cruzada para escolher um hiperparâmetro sem tocar no teste.

A parte 4 é o exercício em duplas da aula.

___
<div id="indice"></div>

## Índice

- [A base](#base)
- [1. Curva ROC e AUC](#roc)
- [2. Treino e teste](#treino)
- [3. Validação cruzada](#cv)
- [4. Exercício: com mais dados, o sobreajuste diminui?](#exercicio)

___
<div id="base"></div>

# A base

**Atenção: a base é simulada.** Ela tem as mesmas colunas da base da aula 20
(ações revisionais de financiamento de veículo), mas são outros 1.000
processos, sorteados com efeitos mais fortes: aqui o objetivo é **prever** a
sentença, e um modelo que erra pouco deixa as contas mais claras. É pública e
lida direto da internet. Os resultados ilustram os métodos e **não dizem nada
sobre processos reais**.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from plotnine import *
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import KFold

processos = pd.read_csv("https://raw.githubusercontent.com/jtrecenti/202662-cdad2/main/dados/processos_acordo.csv")
processos.shape

[Volta ao Índice](#indice)

___
<div id="roc"></div>

# 1. Curva ROC e AUC

O modelo da aula 21: a probabilidade de sentença favorável ao autor, usada pelo
banco para decidir onde oferecer acordo.

In [ ]:
formula = "sentenca_favoravel ~ taxa_juros + escritorio_massa + pericia + juizado + idade_autor"
logit = smf.logit(formula, data=processos).fit()
processos["prob"] = logit.predict(processos)

`roc_curve()` calcula, para cada corte possível, a taxa de falsos positivos
(dos que o banco ganharia, em quantos ofereceu acordo) e a sensibilidade (dos
que perderia, em quantos ofereceu).

In [ ]:
fpr, sens, cortes = roc_curve(processos["sentenca_favoravel"], processos["____"])
roc = pd.DataFrame({"falsos_positivos": fpr, "sensibilidade": sens, "corte": cortes})

(ggplot(roc, aes(x="falsos_positivos", y="sensibilidade"))
 + geom_line(color="seagreen", size=1.2)
 + geom_abline(linetype="dashed")          # o modelo que sorteia
 + coord_equal()
 + theme_minimal())

In [ ]:
roc_auc_score(processos["sentenca_favoravel"], processos["prob"])

**Pergunta.** Interprete a AUC deste modelo. Ela é maior ou menor que a do
NatJus no Projeto 06 (0,80)? O que isso quer dizer?

*(responda aqui)*

[Volta ao Índice](#indice)

___
<div id="treino"></div>

# 2. Treino e teste

Agora o objetivo é **prever o valor da condenação** a partir do valor da causa,
nas sentenças favoráveis. Para ver o sobreajuste com clareza, o modelo é
ajustado em só **15 processos** (o treino), e avaliado nos outros (o teste),
como nos slides.

In [ ]:
favoraveis = processos.dropna(subset=["valor_condenacao"])
favoraveis = favoraveis[favoraveis["valor_causa"] <= 80]

treino = favoraveis.sample(15, random_state=47)
teste = favoraveis.drop(treino.index)
# o teste só com valores de causa dentro da faixa do treino
teste = teste[teste["valor_causa"].between(treino["valor_causa"].min(), treino["valor_causa"].max())]
len(treino), len(teste)

O modelo é um **polinômio** do valor da causa: grau 1 é a reta, grau 2 uma
parábola, e assim por diante. Quanto maior o grau, mais a curva pode se dobrar.
O erro é a **raiz do erro quadrático médio**, em R\$ mil: o tamanho típico do
erro de previsão.

In [ ]:
def ajusta(dados, grau):
    """Ajusta o polinômio e devolve uma função que prevê a condenação."""
    media, dp = dados["valor_causa"].mean(), dados["valor_causa"].std()
    coefs = np.polyfit((dados["valor_causa"] - media) / dp, dados["valor_condenacao"], grau)
    return lambda novos: np.polyval(coefs, (novos["valor_causa"] - media) / dp)

def erro(prever, dados):
    return np.sqrt(np.mean((prever(dados) - dados["valor_condenacao"]) ** 2))

In [ ]:
linhas = []
for grau in range(0, 9):
    prever = ajusta(treino, grau)
    linhas.append({"grau": grau,
                   "erro_treino": erro(prever, treino),
                   "erro_teste": erro(prever, ____)})
erros = pd.DataFrame(linhas)
erros.round(2)

In [ ]:
(ggplot(erros.melt(id_vars="grau", var_name="base", value_name="erro"),
        aes(x="grau", y="erro", color="base"))
 + geom_line(size=1.2) + geom_point(size=3)
 + coord_cartesian(ylim=(0, 10))
 + labs(x="grau do polinômio", y="erro típico (R$ mil)")
 + theme_minimal())

**Perguntas.**

1. O que acontece com o erro no treino quando o grau aumenta? E com o erro no
   teste?
2. Qual grau tem o menor erro no teste? Por que não podemos usar o teste para
   escolher o grau?

*(responda aqui)*

[Volta ao Índice](#indice)

___
<div id="cv"></div>

# 3. Validação cruzada

A validação cruzada escolhe o grau **só com o treino**: divide os 15 processos
em 5 partes, ajusta em 4 e mede o erro na quinta, cinco vezes, e tira a média.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=3)

linhas = []
for grau in range(0, 9):
    erros_rodadas = []
    for i_ajuste, i_validacao in kf.split(treino):
        prever = ajusta(treino.iloc[i_ajuste], grau)
        erros_rodadas.append(erro(prever, treino.iloc[____]))
    linhas.append({"grau": grau, "erro_cv": np.mean(erros_rodadas)})
cv = pd.DataFrame(linhas)
cv.round(2)

In [ ]:
grau_escolhido = cv.loc[cv["erro_cv"].idxmin(), "grau"]
prever = ajusta(treino, grau_escolhido)
grau_escolhido, erro(prever, teste)    # o teste usado uma vez só, no fim

**Pergunta.** Que grau a validação cruzada escolheu? Ele coincide com o de
menor erro no teste da parte 2?

*(responda aqui)*

[Volta ao Índice](#indice)

___
<div id="exercicio"></div>

# 4. Exercício: com mais dados, o sobreajuste diminui?

Em duplas. Repita a parte 2 com **200 processos de treino** em vez de 15.

In [ ]:
treino200 = favoraveis.sample(____, random_state=47)
teste200 = favoraveis.drop(treino200.index)
teste200 = teste200[teste200["valor_causa"].between(treino200["valor_causa"].min(),
                                                    treino200["valor_causa"].max())]

linhas = []
for grau in range(0, 9):
    prever = ajusta(treino200, grau)
    linhas.append({"grau": grau,
                   "erro_treino": erro(prever, ____),
                   "erro_teste": erro(prever, ____)})
pd.DataFrame(linhas).round(2)

**Perguntas.**

1. Compare com a parte 2: o erro no teste do grau 8 ainda explode?
2. Por que mais dados diminuem o sobreajuste?
3. Em uma frase: que erro um escritório deveria pedir a quem vende um modelo de
   previsão de condenação, e por quê?

*(responda aqui)*

[Volta ao Índice](#indice)